# JOBSHEET 06 DL: Teknik Segmentasi Gambar 2 (Segmentasi Instans dan Segmentasi Promptable)
Mata kuliah Visi Komputer, Politeknik Negeri Semarang. Dosen: Ir. Prayitno, S.ST., M.T., Ph.D.

Isi: D1 data instans Penn-Fudan, D2 Mask R-CNN pra-latih COCO, D3 fine-tuning ringan Mask R-CNN,
D4 evaluasi mask (IoU dan AP), D5 Segment Anything (SAM) dengan prompt titik dan kotak,
D6 mini-proyek penghapus latar otomatis (deteksi lalu SAM) pada foto hewan peliharaan.

**Cara pakai di Google Colab:** Runtime > Change runtime type > T4 GPU, lalu Runtime > Run all.

| Bagian | Perkiraan waktu Colab T4 (mode PENUH) | Waktu nyata di CPU box (mode CEPAT) |
|---|---|---|
| Instalasi + unduh Penn-Fudan (51 MB) + bobot | 2 sampai 4 menit | lihat output sel |
| D2 inferensi Mask R-CNN pra-latih | kurang dari 1 menit | lihat output sel |
| D3 fine-tuning Mask R-CNN | sekitar 6 sampai 10 menit | lihat output sel |
| D4 evaluasi | kurang dari 1 menit | lihat output sel |
| D5 SAM ViT-B (unduh 375 MB + inferensi) | 2 sampai 3 menit | lihat output sel |
| D6 mini-proyek | 1 sampai 3 menit | lihat output sel |

Perkiraan waktu Colab adalah estimasi, bukan hasil pengukuran. Tanpa GPU, notebook otomatis memakai mode CEPAT.

In [ ]:
%pip install -q transformers

In [ ]:
# === Persiapan 2: impor, cek GPU, seed, konfigurasi ===
import os, time, random, json, zipfile, urllib.request, gc
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch, torchvision
import torchvision.transforms.functional as TF

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CEPAT = DEVICE == "cpu"
print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__)
print("Perangkat:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU saja")
CFG = dict(
    N_TEST    = 20 if CEPAT else 50,    # citra uji Penn-Fudan
    N_TRAIN   = 60 if CEPAT else 120,   # citra latih untuk fine-tuning
    EPOCHS    = 1 if CEPAT else 6,
    MIN_SIZE  = 320 if CEPAT else 512,  # sisi terpendek citra di dalam Mask R-CNN
    MAX_SIZE  = 544 if CEPAT else 853,
    N_SAM     = 6 if CEPAT else 20,     # citra uji untuk eksperimen SAM
    N_PROYEK  = 8 if CEPAT else 30,     # foto hewan untuk mini-proyek
)
print("Mode:", "CEPAT (CPU)" if CEPAT else "PENUH (GPU)", CFG)
FIG_DIR = "figures"; os.makedirs(FIG_DIR, exist_ok=True)
WAKTU = {}

## Praktikum D1: Mengenal Data Segmentasi Instans (Penn-Fudan Pedestrian)
Penn-Fudan Pedestrian Database (Wang dkk., 2007): 170 citra, 345 pejalan kaki berlabel, setiap orang memiliki masker sendiri
(nilai piksel 0 = latar, 1, 2, 3, ... = ID orang). Sumber resmi: https://www.cis.upenn.edu/~jshi/ped_html/

**Catatan lisensi:** halaman resmi dan `readme.txt` tidak mencantumkan lisensi terbuka; hak cipta tetap pada penulis.
Gunakan hanya untuk belajar, unduh langsung dari sumber resmi, jangan unggah ulang datanya, dan sitasi makalah aslinya.

In [ ]:
# === D1.1 Unduh dan ekstrak Penn-Fudan ===
t0 = time.time()
os.makedirs("data", exist_ok=True)
if not os.path.isdir("data/PennFudanPed"):
    urllib.request.urlretrieve("https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip", "data/PennFudanPed.zip")
    with zipfile.ZipFile("data/PennFudanPed.zip") as z:
        z.extractall("data")
PF = "data/PennFudanPed"
NAMA = sorted(os.listdir(f"{PF}/PNGImages"))
print("Jumlah citra:", len(NAMA), "| waktu %.1f s" % (time.time() - t0))

In [ ]:
# === D1.2 Dataset instans: boxes, labels, masks per orang ===
class PennFudan(torch.utils.data.Dataset):
    def __init__(self, nama, augment=False):
        self.nama, self.augment = nama, augment

    def __len__(self):
        return len(self.nama)

    def muat(self, i):
        n = self.nama[i]
        img = Image.open(f"{self.root}/PNGImages/{n}").convert("RGB")
        m = np.array(Image.open(f"{self.root}/PedMasks/{n[:-4]}_mask.png"))
        ids = np.unique(m)
        ids = ids[ids != 0]  # abaikan latar belakang (0)
        masks = (m[None] == ids[:, None, None])
        return img, masks

    def __getitem__(self, i):
        img, masks = self.muat(i)
        x = TF.to_tensor(img)
        masks = torch.from_numpy(masks.astype(np.uint8))
        if self.augment and random.random() < 0.5:
            x, masks = x.flip(-1), masks.flip(-1)
        boxes = []
        for mk in masks:
            ys, xs = torch.where(mk > 0)
            boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])
        target = {"boxes": torch.tensor(boxes, dtype=torch.float32),
                  "labels": torch.ones(len(masks), dtype=torch.int64),   # 1 = pejalan kaki
                  "masks": masks}
        return x, target

rng = random.Random(0); acak = NAMA[:]; rng.shuffle(acak)
NAMA_TEST, NAMA_TRAIN = acak[:CFG["N_TEST"]], acak[50:50 + CFG["N_TRAIN"]]   # 50 citra pertama dicadangkan untuk uji
ds_all = PennFudan(NAMA)
jml = [len(ds_all.muat(i)[1]) for i in range(len(NAMA))]
print("Total pejalan kaki berlabel:", sum(jml), "| rata-rata per citra: %.2f | maks: %d" % (np.mean(jml), max(jml)))
ds_train = PennFudan(NAMA_TRAIN, augment=True); ds_test = PennFudan(NAMA_TEST)
x, t = ds_test[0]
print("x:", tuple(x.shape), "| boxes:", tuple(t["boxes"].shape), "| masks:", tuple(t["masks"].shape))

In [ ]:
# === D1.3 Visualisasi masker instans: setiap orang diberi warna berbeda ===
WARNA = plt.cm.tab10(np.arange(10))[:, :3]
def overlay(img, masks, boxes=None, alpha=0.5, judul=None, ax=None):
    arr = np.array(img).astype(float) / 255
    for k, mk in enumerate(masks):
        mk = np.asarray(mk).astype(bool)
        arr[mk] = (1 - alpha) * arr[mk] + alpha * WARNA[k % 10]
    ax.imshow(arr)
    if boxes is not None:
        for k, b in enumerate(boxes):
            b = [float(v) for v in b]
            ax.add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, color=WARNA[k % 10], lw=2))
    if judul: ax.set_title(judul, fontsize=9)
    ax.axis("off")

fig, ax = plt.subplots(1, 4, figsize=(16, 5))
for j in range(4):
    img, masks = ds_test.muat(j); _, t = ds_test[j]
    overlay(img, masks, t["boxes"], judul=f"{NAMA_TEST[j]}: {len(masks)} orang", ax=ax[j])
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/js06_d1_contoh_instans.png", dpi=110); plt.show()

### **Tugas kecil D1:** Jelaskan perbedaan keluaran segmentasi semantik (JS05) dan segmentasi instans pada citra yang berisi tiga orang.

In [ ]:
# === TUCIL D1: Perbandingan Segmentasi Semantik vs Instans ===

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import torch
import torchvision.transforms.functional as TF
from PIL import Image

def semantic_mask(masks):
    """
    Gabungkan semua mask instans → satu mask semantik biner.
    Semua piksel 'orang' = 1, latar = 0. Tidak ada info individu.
    """
    if len(masks) == 0:
        return np.zeros_like(np.asarray(masks[0]) if masks else (1, 1), dtype=np.uint8)
    combined = np.zeros_like(np.asarray(masks[0]), dtype=np.uint8)
    for mk in masks:
        combined = np.maximum(combined, np.asarray(mk).astype(np.uint8))
    return combined  # shape: (H, W), nilai 0 atau 1

def overlay_semantic(img, sem_mask, judul=None, ax=None):
    """Visualisasi segmentasi semantik: satu warna untuk semua orang."""
    arr = np.array(img).astype(float) / 255
    mk_bool = sem_mask.astype(bool)
    # Semua piksel 'orang' diberi warna tunggal (merah)
    arr[mk_bool] = (1 - 0.5) * arr[mk_bool] + 0.5 * np.array([1.0, 0.2, 0.2])
    ax.imshow(arr)
    patch = mpatches.Patch(color=[1.0, 0.2, 0.2], label="person (kelas 1)")
    ax.legend(handles=[patch], fontsize=7, loc="upper right")
    if judul:
        ax.set_title(judul, fontsize=9)
    ax.axis("off")

def overlay_instance(img, masks, boxes, judul=None, ax=None):
    """Visualisasi segmentasi instans: setiap orang warna berbeda + bbox."""
    arr = np.array(img).astype(float) / 255
    patches = []
    for k, mk in enumerate(masks):
        mk_bool = np.asarray(mk).astype(bool)
        warna = WARNA[k % 10]
        arr[mk_bool] = (1 - 0.5) * arr[mk_bool] + 0.5 * warna
        b = [float(v) for v in boxes[k]]
        ax.add_patch(plt.Rectangle(
            (b[0], b[1]), b[2] - b[0], b[3] - b[1],
            fill=False, edgecolor=warna, lw=2
        ))
        patches.append(mpatches.Patch(color=warna, label=f"Inst #{k+1}"))
    ax.imshow(arr)
    ax.legend(handles=patches, fontsize=7, loc="upper right")
    if judul:
        ax.set_title(judul, fontsize=9)
    ax.axis("off")

# --- Pilih 3 citra yang masing-masing berisi ≥3 orang ---
idx_kandidat = [
    i for i in range(len(ds_test))
    if len(ds_test.muat(i)[1]) >= 3
][:3]

# Fallback: ambil 3 citra pertama jika tidak cukup yang berisi ≥3 orang
if len(idx_kandidat) < 3:
    idx_kandidat = list(range(min(3, len(ds_test))))

print(f"Citra dipilih (idx): {idx_kandidat}")
for i in idx_kandidat:
    img, masks = ds_test.muat(i)
    print(f"  {NAMA_TEST[i]}: {len(masks)} orang")

# --- Plot: 3 baris (per citra) × 3 kolom (asli | semantik | instans) ---
fig, axes = plt.subplots(
    len(idx_kandidat), 3,
    figsize=(14, 4.5 * len(idx_kandidat))
)
if len(idx_kandidat) == 1:
    axes = [axes]  # pastikan iterable

for row, idx in enumerate(idx_kandidat):
    img, masks = ds_test.muat(idx)
    _, target = ds_test[idx]
    boxes = target["boxes"]
    n_orang = len(masks)

    sem = semantic_mask(masks)

    # Kolom 0 — Citra Asli
    axes[row][0].imshow(img)
    axes[row][0].set_title(f"Asli\n{NAMA_TEST[idx]}", fontsize=9)
    axes[row][0].axis("off")

    # Kolom 1 — Segmentasi Semantik
    overlay_semantic(
        img, sem,
        judul=f"Semantik\n→ 1 kelas, {n_orang} orang tak terbedakan",
        ax=axes[row][1]
    )

    # Kolom 2 — Segmentasi Instans
    overlay_instance(
        img, masks, boxes,
        judul=f"Instans\n→ {n_orang} mask + bbox terpisah",
        ax=axes[row][2]
    )

plt.suptitle(
    "D1 — Perbandingan Segmentasi Semantik vs Instans\n"
    "(3 citra, masing-masing ≥3 pejalan kaki)",
    fontsize=11, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/js06_tucil_d1_semantik_vs_instans.png", dpi=110, bbox_inches="tight")
plt.show()
print("Gambar disimpan.")

## Praktikum D2: Mask R-CNN Pra-Latih COCO (tanpa pelatihan)
Mask R-CNN (He dkk., 2017) menambahkan cabang prediksi masker pada detektor Faster R-CNN. Bobot torchvision dilatih pada COCO
(80 kelas, kelas 1 = *person*), sehingga dapat langsung mendeteksi pejalan kaki.

In [ ]:
# === D2.1 Muat Mask R-CNN pra-latih dan lakukan inferensi ===
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights

W = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
mrcnn_coco = maskrcnn_resnet50_fpn(weights=W, min_confidence=CFG["MIN_SIZE"]).to(DEVICE).eval()
KATEGORI = W.meta["categories"]

@torch.no_grad()
def deteksi(model, x, label_orang=1, skor_min=0.5):
    out = model([x.to(DEVICE)])[0]
    keep = (out["labels"] == label_orang) & (out["scores"] >= skor_min)
    return {k: v[keep].cpu() for k, v in out.items()}

t0 = time.time()
fig, ax = plt.subplots(1, 4, figsize=(16, 5))
for j in range(4):
    x, t = ds_test[j]; img, _ = ds_test.muat(j)
    o = deteksi(mrcnn_coco, x)
    overlay(img, (o["masks"][:, 0] > 0.5).numpy(), o["boxes"], judul=f"prediksi {len(o['scores'])} orang (GT {len(t['labels'])})", ax=ax[j])
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/js06_d2_maskrcnn_coco.png", dpi=110); plt.show()

### **Tugas kecil D2:** Ubah `skor_min` menjadi 0,3 dan 0,8. Berapa jumlah deteksi yang berubah? Apa risikonya bila ambang terlalu rendah?

In [ ]:
# === TUCIL D2: Eksperimen skor_min 0.3 vs 0.5 vs 0.8 ===

THRESHOLDS = [0.3, 0.5, 0.8]
N_CITRA = 4

fig, axes = plt.subplots(N_CITRA, len(THRESHOLDS), figsize=(15, 5 * N_CITRA))

ringkasan = {thr: [] for thr in THRESHOLDS}

for j in range(N_CITRA):
    x, t = ds_test[j]
    img, _ = ds_test.muat(j)
    gt_count = len(t["labels"])

    for col, thr in enumerate(THRESHOLDS):
        o = deteksi(mrcnn_coco, x, skor_min=thr)
        n_pred = len(o["scores"])
        ringkasan[thr].append(n_pred)

        masks_bin = (o["masks"][:, 0] > 0.5).numpy() if n_pred > 0 else []
        boxes     = o["boxes"] if n_pred > 0 else []

        overlay(
            img, masks_bin, boxes,
            judul=f"skor_min={thr} → {n_pred} deteksi (GT={gt_count})",
            ax=axes[j][col]
        )

plt.suptitle(
    "Tucil D2 — Pengaruh Threshold Skor terhadap Jumlah Deteksi",
    fontsize=12, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/js06_tucil_d2_threshold.png", dpi=110, bbox_inches="tight")
plt.show()

# --- Tabel ringkasan per threshold ---
print(f"\n{'Citra':<10}", end="")
for thr in THRESHOLDS:
    print(f"  skor≥{thr}", end="")
print()
print("-" * 40)

for j in range(N_CITRA):
    x, t = ds_test[j]
    gt = len(t["labels"])
    print(f"Citra {j+1:<4}(GT={gt})", end="")
    for thr in THRESHOLDS:
        print(f"  {ringkasan[thr][j]:^10}", end="")
    print()

print("-" * 40)
print(f"{'Rata-rata':<10}", end="")
for thr in THRESHOLDS:
    print(f"  {np.mean(ringkasan[thr]):^9.1f}", end="")
print()

# --- Analisis delta deteksi ---
print("\n=== Analisis Delta Deteksi ===")
delta_low  = np.mean(ringkasan[0.3]) - np.mean(ringkasan[0.5])
delta_high = np.mean(ringkasan[0.5]) - np.mean(ringkasan[0.8])
print(f"0.5 → 0.3 : +{delta_low:.1f} deteksi rata-rata (tambahan = kandidat False Positive)")
print(f"0.5 → 0.8 : -{delta_high:.1f} deteksi rata-rata (hilang = berisiko False Negative)")

| `skor_min` | Efek |
| :--- | :--- |
| **0.3** | Deteksi bertambah — model meloloskan prediksi dengan keyakinan rendah |
| **0.5** | Baseline — keseimbangan presisi/recall |
| **0.8** | Deteksi berkurang — hanya prediksi dengan keyakinan sangat tinggi |

Risiko ambang terlalu rendah (0.3):
- False Positive meningkat — benda bukan orang (tiang, tas, bayangan) ikut terdeteksi sebagai pejalan kaki
- Mask overlap — dua mask berbeda bisa mencover orang yang sama → counting salah
Pada aplikasi nyata (CCTV, crowd counting): angka kepadatan jadi tidak akurat

Risiko ambang terlalu tinggi (0.8):
- False Negative meningkat — orang yang sebagian tertutup (occluded) atau blur tidak terdeteksi sama sekali
Berbahaya pada sistem keselamatan (misal: deteksi pejalan kaki di zebra cross)

## Praktikum D4 (dikerjakan sebelum D3): Fungsi Evaluasi Mask (IoU dan AP)
Kita membuat evaluator ringan: setiap prediksi dicocokkan dengan masker GT yang belum terpakai (urut skor tertinggi).
Prediksi dianggap benar (TP) bila IoU masker ≥ ambang. Dari kurva presisi-recall dihitung **AP** (Average Precision).
AP50 memakai ambang 0,5; AP[.50:.95] adalah rata-rata AP pada ambang 0,50 sampai 0,95 (gaya COCO).

In [ ]:
# === D4.1 Evaluator mask: IoU, AP50, AP[.50:.95] ===
def iou_mask(a, b):
    # a: (P,H,W) bool, b: (G,H,W) bool  ->  (P,G)
    a = a.reshape(a.shape[0], -1).float(); b = b.reshape(b.shape[0], -1).float()
    inter = a @ b.T
    return inter / (a.sum(1, keepdim=True) + b.sum(1)[None] - inter).clamp_min(1)

def average_precision(skor, tp, npos):
    if len(skor) == 0: return 0.0, np.array([0.]), np.array([0.])
    o = np.argsort(-np.array(skor)); tp = np.array(tp)[o]
    ctp, cfp = np.cumsum(tp), np.cumsum(1 - tp)
    rec = ctp / max(npos, 1); prec = ctp / np.maximum(ctp + cfp, 1e-9)
    mrec = np.r_[0, rec, 1]; mpre = np.r_[0, prec, 0]
    for i in range(len(mpre) - 2, -1, -1): mpre[i] = max(mpre[i], mpre[i + 1])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])), rec, prec

@torch.no_grad()
def evaluasi_instans(model, ds, label_orang=1, skor_min=0.05):
    model.eval(); semua = []
    for i in range(len(ds)):
        x, t = ds[i]
        o = deteksi(model, x, label_orang, skor_min)
        P = o["masks"][:, 0] > 0.5; G = t["masks"].bool()
        semua.append((o["scores"].numpy(), iou_mask(P, G) if len(P) else torch.zeros(0, len(G)), len(G)))
    npos = sum(s[2] for s in semua); hasil = {}
    for thr in np.arange(0.5, 0.96, 0.05):
        skor, tp, iou_cocok = [], [], []
        for sc, M, g in semua:
            pakai = set()
            for p in np.argsort(-sc):
                skor.append(sc[p]); best, bj = 0, -1
                for j in range(g):
                    if j not in pakai and M[p, j] > best: best, bj = M[p, j].item(), j
                if best >= thr: pakai.add(bj); tp.append(1); iou_cocok.append(best)
                else: tp.append(0)
        ap, rec, prec = average_precision(skor, tp, npos)
        hasil[round(float(thr), 2)] = (ap, rec, prec, iou_cocok)
    return {"AP50": hasil[0.5][0], "AP75": hasil[0.75][0],
            "AP[.50:.95]": float(np.mean([v[0] for v in hasil.values()])),
            "mIoU_cocok@0.5": float(np.mean(hasil[0.5][3])) if hasil[0.5][3] else 0.0,
            "Recall@0.5": float(hasil[0.5][1][-1]), "jumlah_GT": npos, "_pr": hasil[0.5][1:3]}

t0 = time.time()
HASIL = {"Mask R-CNN COCO (tanpa latih)": evaluasi_instans(mrcnn_coco, ds_test)}
WAKTU["Evaluasi COCO zero-shot"] = time.time() - t0
ringkas = lambda m: {k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.items() if not k.startswith("_")}
print(ringkas(HASIL["Mask R-CNN COCO (tanpa latih)"]), "| %.1f s" % WAKTU["Evaluasi COCO zero-shot"])

In [ ]:
import time

# === D4.2 Evaluasi Model Hasil Fine-Tuning dan Perbandingan ===
t0 = time.time()
HASIL["Mask R-CNN fine-tuned Penn-Fudan"] = evaluasi_instans(mrcnn_ft, ds_test)
WAKTU["Evaluasi fine-tuned"] = time.time() - t0

for k, v in HASIL.items():
    print(f"{k:35s}", ringkas(v))

plt.figure(figsize=(5, 4))
for k, v in HASIL.items():
    rec, prec = v["_pr"]
    plt.plot(rec, prec, label=f"{k} (AP50={v['AP50']:.2f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Kurva Presisi-Recall (IoU Masker 0.5)")
plt.legend(fontsize=7)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/js06_d4_kurva_pr.png", dpi=120)
plt.show()


# === D4.3 Visualisasi: GT vs COCO vs Fine-Tuned ===
fig, ax = plt.subplots(3, 3, figsize=(13, 13))

for r, j in enumerate([4, 5, 6]):
    x, t = ds_test[j]
    img, gm = ds_test.muat(j)
    
    # 1. Ground Truth
    overlay(img, gm, t["boxes"], judul="Ground Truth", ax=ax[r, 0])
    
    # 2. COCO Tanpa Latih (Pretrained)
    o_coco = deteksi(mrcnn_coco, x)
    overlay(
        img, 
        (o_coco["masks"][:, 0] > 0.5).numpy(), 
        o_coco["boxes"], 
        judul="COCO Tanpa Latih", 
        ax=ax[r, 1]
    )
    
    # 3. Model Fine-Tuned
    o_ft = deteksi(mrcnn_ft, x)
    overlay(
        img, 
        (o_ft["masks"][:, 0] > 0.5).numpy(), 
        o_ft["boxes"], 
        judul="Fine-Tuned", 
        ax=ax[r, 2]
    )

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/js06_d4_perbandingan.png", dpi=100)
plt.show()

## Praktikum D3: Fine-Tuning Ringan Mask R-CNN untuk Kelas Pejalan Kaki
Kepala klasifikasi kotak dan kepala masker diganti menjadi 2 kelas (latar + pejalan kaki). Backbone ResNet-50 dibekukan
agar pelatihan ringan; hanya RPN, kepala kotak, dan kepala masker yang dilatih.

In [ ]:
# === D3.1 Siapkan model 2 kelas ===
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

set_seed(42)

mrcnn_ft = maskrcnn_resnet50_fpn(weights=None, min_size=CFG["MIN_SIZE"], max_size=CFG["MAX_SIZE"])
in_feat = mrcnn_ft.roi_heads.box_predictor.cls_score.in_features
mrcnn_ft.roi_heads.box_predictor = FastRCNNPredictor(in_feat, 2)  # 2 kelas: latar belakang + pejalan kakim
mrcnn_ft.roi_heads.mask_predictor = MaskRCNNPredictor(256, 256, 2)  # 2 kelas: latar belakang + pejalan kaki
for p in mrcnn_ft.backbone.body.parameters():
    p.requires_grad = False  # beku resnet50 untuk fine-tuning
mrcnn_ft = mrcnn_ft.to(DEVICE)
dilatih = [p for p in mrcnn_ft.parameters() if p.requires_grad]
print("Parameter dilatih: %.2f juta dari %.2f juta" % (sum(p.numel() for p in dilatih) / 1e6, sum(p.numel() for p in mrcnn_ft.parameters()) / 1e6))

In [ ]:
# === D3.2 Loop pelatihan ===
collate = lambda b: tuple(zip(*b))  # untuk DataLoader Mask R-CNN
dl_train = torch.utils.data.DataLoader(ds_train, batch_size=2, shuffle=True, collate_fn=collate)
opt = torch.optim.SGD(dilatih, lr=0.005, momentum=0.9, weight_decay=5e-4)
riwayat = []; t0 = time.time()
for ep in range(CFG["EPOCHS"]):
    mrcnn_ft.train()
    for it, (xs, ts) in enumerate(dl_train):
        xs = [x.to(DEVICE) for x in xs]
        ts = [{k: v.to(DEVICE) for k, v in t.items()} for t in ts]
        losses = mrcnn_ft(xs, ts)       # mode latih: keluaran berupa dict loss
        loss = sum(losses.values())
        opt.zero_grad(); loss.backward(); opt.step()
        riwayat.append({k: v.item() for k, v in losses.items()} | {"total": loss.item()})
        if it % 10 == 0:
            print(f"epoch {ep+1} iterasi {it:3d}  loss total {loss.item():.3f}  mask {losses['loss_mask'].item():.3f}  ({time.time()-t0:.0f} s)")
WAKTU["Fine-tuning Mask R-CNN"] = time.time() - t0
print("Waktu fine-tuning: %.1f s untuk %d iterasi" % (WAKTU["Fine-tuning Mask R-CNN"], len(riwayat)))

In [ ]:
# === D3.3 Kurva loss fine-tuning ===
plt.figure(figsize=(7, 4))
for k in ["total", "loss_classifier", "loss_box_reg", "loss_mask", "loss_objectness"]:
    v = np.array([r[k] for r in riwayat]); w = max(1, len(v) // 10)
    plt.plot(np.convolve(v, np.ones(w) / w, mode="valid"), label=k)
plt.xlabel("Iterasi"); plt.ylabel("Loss (rata-rata bergerak)"); plt.title("Fine-tuning Mask R-CNN pada Penn-Fudan")
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/js06_d3_kurva_finetune.png", dpi=120); plt.show()

In [ ]:
# === D4.2 Evaluasi model hasil fine-tuning dan bandingkan ===
t0 = time.time()
HASIL["Mask R-CNN fine-tuned Penn-Fudan"] = evaluasi_instans(mrcnn_ft, ds_test)
WAKTU["Evaluasi fine-tuned"] = time.time() - t0
for k, v in HASIL.items():
    print(f"{k:35s}", ringkas(v))
plt.figure(figsize=(5, 4))
for k, v in HASIL.items():
    rec, prec = v["_pr"]; plt.plot(rec, prec, label=f"{k} (AP50={v['AP50']:.2f})")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Kurva presisi-recall (IoU masker 0,5)")
plt.legend(fontsize=7); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/js06_d4_kurva_pr.png", dpi=120); plt.show()

In [ ]:
# === D4.3 Visual: GT vs COCO vs fine-tuned ===
fig, ax = plt.subplots(3, 3, figsize=(13, 13))
for r, j in enumerate([4, 5, 6]):
    x, t = ds_test[j]; img, gm = ds_test.muat(j)
    overlay(img, gm, t["boxes"], judul="ground truth", ax=ax[r, 0])
    o = deteksi(mrcnn_coco, x); overlay(img, (o["masks"][:, 0] > 0.5).numpy(), o["boxes"], judul="COCO tanpa latih", ax=ax[r, 1])
    o = deteksi(mrcnn_ft, x);   overlay(img, (o["masks"][:, 0] > 0.5).numpy(), o["boxes"], judul="fine-tuned", ax=ax[r, 2])
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/js06_d4_perbandingan.png", dpi=100); plt.show()

### **Tugas kecil D3/D4:** Latih 2 sampai 3 epoch lagi (atau buka pembekuan `layer4` backbone). Apakah AP50 dan AP[.50:.95] naik?
Mengapa model COCO tanpa pelatihan sudah cukup baik untuk kelas orang?

In [ ]:
# === TUCIL D3/D4: Lanjut latih + unfreeze layer4 backbone ===

import copy

# --- Simpan baseline sebelum lanjut latih ---
ap_baseline = HASIL["Mask R-CNN fine-tuned Penn-Fudan"].copy()
print("=== Baseline (sebelum eksperimen) ===")
print(f"  AP50          : {ap_baseline['AP50']:.4f}")
print(f"  AP[.50:.95]   : {ap_baseline['AP[.50:.95]']:.4f}")
print(f"  mIoU@0.5      : {ap_baseline['mIoU_cocok@0.5']:.4f}")

In [ ]:
# --- Tahap 1: Lanjut latih 3 epoch (backbone tetap beku) ---
EXTRA_EPOCHS = 3
riwayat_lanjut = []
t0 = time.time()

for ep in range(EXTRA_EPOCHS):
    mrcnn_ft.train()
    for it, (xs, ts) in enumerate(dl_train):
        xs = [x.to(DEVICE) for x in xs]
        ts = [{k: v.to(DEVICE) for k, v in t.items()} for t in ts]
        losses = mrcnn_ft(xs, ts)
        loss = sum(losses.values())
        opt.zero_grad(); loss.backward(); opt.step()
        riwayat_lanjut.append({k: v.item() for k, v in losses.items()} | {"total": loss.item()})
        if it % 10 == 0:
            print(f"[lanjut] epoch {ep+1} it {it:3d}  total {loss.item():.3f}  mask {losses['loss_mask'].item():.3f}")

WAKTU["Lanjut latih +3 epoch"] = time.time() - t0
print(f"\nWaktu: {WAKTU['Lanjut latih +3 epoch']:.1f} s")

HASIL["Lanjut latih +3 epoch (frozen)"] = evaluasi_instans(mrcnn_ft, ds_test)

In [ ]:
# --- Tahap 2: Unfreeze layer4 backbone, LR lebih kecil ---
for name, param in mrcnn_ft.backbone.body.named_parameters():
    if name.startswith("layer4"):
        param.requires_grad = True

dilatih_unfreeze = [p for p in mrcnn_ft.parameters() if p.requires_grad]
print(f"Parameter dilatih setelah unfreeze layer4: "
      f"{sum(p.numel() for p in dilatih_unfreeze)/1e6:.2f} juta")

# LR lebih kecil untuk layer yang baru dibuka — hindari catastrophic forgetting
opt_unfreeze = torch.optim.SGD(
    dilatih_unfreeze, lr=0.001,   # 5x lebih kecil dari sebelumnya
    momentum=0.9, weight_decay=5e-4
)

UNFREEZE_EPOCHS = 3
riwayat_unfreeze = []
t0 = time.time()

for ep in range(UNFREEZE_EPOCHS):
    mrcnn_ft.train()
    for it, (xs, ts) in enumerate(dl_train):
        xs = [x.to(DEVICE) for x in xs]
        ts = [{k: v.to(DEVICE) for k, v in t.items()} for t in ts]
        losses = mrcnn_ft(xs, ts)
        loss = sum(losses.values())
        opt_unfreeze.zero_grad(); loss.backward(); opt_unfreeze.step()
        riwayat_unfreeze.append({k: v.item() for k, v in losses.items()} | {"total": loss.item()})
        if it % 10 == 0:
            print(f"[unfreeze] epoch {ep+1} it {it:3d}  total {loss.item():.3f}  mask {losses['loss_mask'].item():.3f}")

WAKTU["Unfreeze layer4 +3 epoch"] = time.time() - t0
print(f"\nWaktu: {WAKTU['Unfreeze layer4 +3 epoch']:.1f} s")

HASIL["Unfreeze layer4 +3 epoch"] = evaluasi_instans(mrcnn_ft, ds_test)

In [ ]:
# --- Kurva loss gabungan semua fase ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fases = [
    ("Fine-tune awal (frozen)",  riwayat,          "C0"),
    ("Lanjut +3 epoch (frozen)", riwayat_lanjut,   "C1"),
    ("Unfreeze layer4",          riwayat_unfreeze,  "C2"),
]

for ax_i, (judul, rwyt, warna) in zip(axes, fases):
    for k, ls in [("total","solid"),("loss_mask","dashed"),("loss_classifier","dotted")]:
        v = np.array([r[k] for r in rwyt])
        w = max(1, len(v) // 8)
        ax_i.plot(
            np.convolve(v, np.ones(w)/w, mode="valid"),
            label=k, linestyle=ls, color=warna if k == "total" else None
        )
    ax_i.set_title(judul, fontsize=9)
    ax_i.set_xlabel("Iterasi")
    ax_i.set_ylabel("Loss")
    ax_i.legend(fontsize=7)
    ax_i.grid(alpha=0.3)

plt.suptitle("Kurva Loss Semua Fase Training", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/js06_tucil_d34_loss.png", dpi=110, bbox_inches="tight")
plt.show()

In [ ]:
# --- Tabel perbandingan semua model ---
LABEL_URUT = [
    "Mask R-CNN COCO (tanpa latih)",
    "Mask R-CNN fine-tuned Penn-Fudan",
    "Lanjut latih +3 epoch (frozen)",
    "Unfreeze layer4 +3 epoch",
]

metrik = ["AP50", "AP75", "AP[.50:.95]", "mIoU_cocok@0.5", "Recall@0.5"]
header = f"{'Model':<40}" + "".join(f"{m:>18}" for m in metrik)
print(header)
print("-" * (40 + 18 * len(metrik)))

for label in LABEL_URUT:
    if label not in HASIL:
        continue
    v = HASIL[label]
    row = f"{label:<40}" + "".join(f"{v[m]:>18.4f}" for m in metrik)
    print(row)

# --- Kurva PR semua model ---
plt.figure(figsize=(6, 5))
for label in LABEL_URUT:
    if label not in HASIL:
        continue
    rec, prec = HASIL[label]["_pr"]
    ap = HASIL[label]["AP50"]
    plt.plot(rec, prec, label=f"{label}\n(AP50={ap:.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Kurva PR — Semua Fase (IoU Mask ≥ 0.5)")
plt.legend(fontsize=7, loc="upper right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/js06_tucil_d34_kurva_pr.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# --- Visualisasi prediksi model final vs GT ---
fig, ax = plt.subplots(3, 2, figsize=(10, 13))

for r, j in enumerate([4, 5, 6]):
    x, t = ds_test[j]
    img, gm = ds_test.muat(j)

    # Ground truth
    overlay(img, gm, t["boxes"],
            judul=f"Ground Truth ({len(gm)} orang)",
            ax=ax[r][0])

    # Prediksi model setelah unfreeze layer4
    o = deteksi(mrcnn_ft, x)
    overlay(img,
            (o["masks"][:, 0] > 0.5).numpy(),
            o["boxes"],
            judul=f"Unfreeze layer4 → {len(o['scores'])} deteksi",
            ax=ax[r][1])

plt.suptitle("Tucil D3/D4 — Hasil Akhir Model Fine-Tuned (Unfreeze layer4)",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/js06_tucil_d34_visual_final.png", dpi=100, bbox_inches="tight")
plt.show()

## Praktikum D5: Segment Anything (SAM) dengan Prompt Titik dan Kotak
SAM (Kirillov dkk., 2023) adalah model *promptable*: diberi petunjuk (titik, kotak), ia menghasilkan masker objek apa pun
tanpa pelatihan ulang. Kita memakai **facebook/sam-vit-base** (lisensi Apache-2.0) melalui Hugging Face Transformers dan
membandingkan tiga jenis prompt: (a) satu titik di dalam objek, (b) kotak GT, (c) kotak dari Mask R-CNN (deteksi lalu segmentasi).

In [ ]:
# === D5.1 Muat SAM ===
del mrcnn_ft; gc.collect()
from transformers import SamModel, SamProcessor
import transformers; transformers.logging.set_verbosity_error()
t0 = time.time()
sam_proc = SamProcessor.from_pretrained("facebook/sam-vit-base")
sam = SamModel.from_pretrained("facebook/sam-vit-base").to(DEVICE).eval()
print("Parameter SAM ViT-B: %.1f juta | waktu muat %.1f s" % (sum(p.numel() for p in sam.parameters()) / 1e6, time.time() - t0))

@torch.no_grad()
def sam_embedding(img):
    # encoder gambar SAM (bagian terberat) cukup dijalankan SEKALI per citra
    inp = sam_proc(img, return_tensors="pt").to(DEVICE)
    return sam.get_image_embeddings(inp["pixel_values"])

@torch.no_grad()
def sam_segmentasi(img, points=None, boxes=None, multimask=False, emb=None):
    # points: list per objek, tiap objek berisi daftar titik [[x,y], ...]; boxes: list [[x0,y0,x1,y1], ...]
    kw = {}
    if points is not None:
        kw["input_points"] = [points]; kw["input_labels"] = [[[1] * len(p) for p in points]]
    if boxes is not None:
        kw["input_boxes"] = [[list(map(float, b)) for b in boxes]]
    inp = sam_proc(img, return_tensors="pt", **kw).to(DEVICE)
    if emb is None:
        emb = sam.get_image_embeddings(inp["pixel_values"])
    inp.pop("pixel_values")
    out = sam(image_embeddings=emb, **inp, multimask_output=multimask)
    masks = sam_proc.image_processor.post_process_masks(out.pred_masks.cpu(), inp["original_sizes"].cpu(),
                                                        inp["reshaped_input_sizes"].cpu())[0]   # (obj, k, H, W)
    skor = out.iou_scores[0].cpu()                                                               # (obj, k)
    terbaik = skor.argmax(1)
    return masks[torch.arange(len(masks)), terbaik].bool(), skor.max(1).values

def titik_dalam(mk, fraksi=(0.5,)):
    # titik prompt: piksel masker terdekat dengan sumbu tengah objek pada ketinggian tertentu
    ys, xs = np.where(mk); cx = xs.mean(); y0, y1 = ys.min(), ys.max(); hasil = []
    for f in fraksi:
        cy = y0 + f * (y1 - y0); k = np.argmin((ys - cy) ** 2 + (xs - cx) ** 2)
        hasil.append([int(xs[k]), int(ys[k])])
    return hasil

In [ ]:
# === D5.2 Eksperimen tiga jenis prompt ===
iou_prompt = {"1 titik per orang": [], "3 titik per orang": [], "kotak GT": [], "kotak Mask R-CNN": [],
              "masker Mask R-CNN (pembanding)": []}
contoh = None; t0 = time.time()
for j in range(CFG["N_SAM"]):
    x, t = ds_test[j]; img, gm = ds_test.muat(j); G = torch.from_numpy(gm)
    emb = sam_embedding(img)                                        # sekali per citra
    pts1 = [titik_dalam(mk) for mk in gm]
    pts3 = [titik_dalam(mk, (0.25, 0.5, 0.75)) for mk in gm]
    m_p1, _ = sam_segmentasi(img, points=pts1, multimask=True, emb=emb)   # 1 titik: ambigu, pilih skor tertinggi
    m_p3, _ = sam_segmentasi(img, points=pts3, emb=emb)
    m_bx, _ = sam_segmentasi(img, boxes=t["boxes"].tolist(), emb=emb)
    iou_prompt["1 titik per orang"] += torch.diag(iou_mask(m_p1, G)).tolist()
    iou_prompt["3 titik per orang"] += torch.diag(iou_mask(m_p3, G)).tolist()
    iou_prompt["kotak GT"] += torch.diag(iou_mask(m_bx, G)).tolist()
    o = deteksi(mrcnn_coco, x)
    if len(o["boxes"]):
        m_det, _ = sam_segmentasi(img, boxes=o["boxes"].tolist(), emb=emb)
        iou_prompt["kotak Mask R-CNN"] += iou_mask(m_det, G).max(0).values.tolist()          # IoU terbaik per orang GT
        iou_prompt["masker Mask R-CNN (pembanding)"] += iou_mask(o["masks"][:, 0] > 0.5, G).max(0).values.tolist()
    else:
        iou_prompt["kotak Mask R-CNN"] += [0.0] * len(G); iou_prompt["masker Mask R-CNN (pembanding)"] += [0.0] * len(G)
    if j == 0:
        contoh = (img, gm, pts1, m_p1, pts3, m_p3, t["boxes"], m_bx)
WAKTU["SAM eksperimen"] = time.time() - t0
HASIL_SAM = {k: (float(np.mean(v)), len(v)) for k, v in iou_prompt.items()}
for k, (m, n) in HASIL_SAM.items():
    print(f"{k:35s} IoU rata-rata = {m:.4f}  (n = {n} orang)")
print("Waktu: %.1f s untuk %d citra" % (WAKTU["SAM eksperimen"], CFG["N_SAM"]))

In [ ]:
# === D5.3 Visualisasi prompt SAM ===
img, gm, pts1, m_p1, pts3, m_p3, bxs, m_bx = contoh
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
overlay(img, gm, judul="ground truth", ax=ax[0])
overlay(img, m_p1.numpy(), judul="SAM, prompt 1 titik per orang", ax=ax[1])
overlay(img, m_p3.numpy(), judul="SAM, prompt 3 titik per orang", ax=ax[2])
for a, P in ((ax[1], pts1), (ax[2], pts3)):
    for obj in P:
        for p in obj: a.plot(p[0], p[1], "*", color="yellow", ms=14, mec="k")
overlay(img, m_bx.numpy(), bxs, judul="SAM, prompt kotak", ax=ax[3])
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/js06_d5_sam_prompt.png", dpi=110); plt.show()

### **Tugas kecil D5:** Tambahkan titik negatif (label 0) di latar yang ikut tersegmentasi. Bagaimana masker berubah?
Mengapa satu titik bersifat ambigu (bisa berarti rok, badan, atau seluruh orang), dan bagaimana tiga titik atau kotak mengurangi ambiguitas itu?

In [ ]:
# === TUCIL D5: Titik negatif (label 0) untuk koreksi segmentasi ===

def sam_segmentasi_neg(img, points_pos, points_neg, emb=None, multimask=False):
    """
    Segmentasi SAM dengan titik positif (label=1) DAN negatif (label=0).
    points_pos: list per objek [[x,y], ...]
    points_neg: list titik negatif GLOBAL yang berlaku untuk semua objek [[x,y], ...]
    """
    n_obj = len(points_pos)

    # Gabungkan: tiap objek dapat titik positifnya + semua titik negatif
    all_points = []
    all_labels = []
    for pos in points_pos:
        obj_pts = pos + points_neg                        # gabung positif + negatif
        obj_lbl = [1] * len(pos) + [0] * len(points_neg) # label sesuai
        all_points.append(obj_pts)
        all_labels.append(obj_lbl)

    kw = {
        "input_points": [all_points],
        "input_labels": [all_labels]
    }
    inp = sam_proc(img, return_tensors="pt", **kw).to(DEVICE)
    if emb is None:
        emb = sam.get_image_embeddings(inp["pixel_values"])
    inp.pop("pixel_values")

    out = sam(image_embeddings=emb, **inp, multimask_output=multimask)
    masks = sam_proc.image_processor.post_process_masks(
        out.pred_masks.cpu(),
        inp["original_sizes"].cpu(),
        inp["reshaped_input_sizes"].cpu()
    )[0]
    skor = out.iou_scores[0].cpu()
    terbaik = skor.argmax(1)
    return masks[torch.arange(len(masks)), terbaik].bool(), skor.max(1).values


def temukan_titik_negatif(mask_all, img_shape, n=3, margin=10):
    """
    Cari piksel latar belakang yang JAUH dari semua mask → kandidat titik negatif.
    Strategi: ambil piksel non-mask di tepi/sudut gambar.
    """
    H, W = img_shape[:2]
    mask_union = np.zeros((H, W), dtype=bool)
    for mk in mask_all:
        mask_union |= np.asarray(mk).astype(bool)

    # Area latar: bukan bagian dari mask manapun
    bg = ~mask_union

    # Pilih dari sudut-sudut gambar (pasti latar)
    kandidat = []
    sudut = [
        (margin, margin),              # kiri atas
        (margin, W - margin),          # kanan atas
        (H - margin, margin),          # kiri bawah
        (H - margin, W - margin),      # kanan bawah
        (H // 2, margin),              # kiri tengah
        (H // 2, W - margin),          # kanan tengah
    ]
    for (y, x) in sudut:
        if 0 <= y < H and 0 <= x < W and bg[y, x]:
            kandidat.append([x, y])   # format SAM: [x, y]

    return kandidat[:n]


def overlay_dengan_prompt(img, masks, pts_pos=None, pts_neg=None,
                           boxes=None, judul=None, ax=None, alpha=0.5):
    """Overlay mask + tampilkan titik positif (bintang kuning) dan negatif (silang merah)."""
    arr = np.array(img).astype(float) / 255
    for k, mk in enumerate(masks):
        mk_bool = np.asarray(mk).astype(bool)
        arr[mk_bool] = (1 - alpha) * arr[mk_bool] + alpha * WARNA[k % 10]

    ax.imshow(arr)

    if pts_pos is not None:
        for obj_pts in pts_pos:
            for p in obj_pts:
                ax.plot(p[0], p[1], "*", color="yellow", ms=14, mec="black", mew=1)

    if pts_neg is not None:
        for p in pts_neg:
            ax.plot(p[0], p[1], "x", color="red", ms=14, mew=3)

    if boxes is not None:
        for k, b in enumerate(boxes):
            b = [float(v) for v in b]
            ax.add_patch(plt.Rectangle(
                (b[0], b[1]), b[2] - b[0], b[3] - b[1],
                fill=False, edgecolor=WARNA[k % 10], lw=2
            ))

    if judul:
        ax.set_title(judul, fontsize=9)
    ax.axis("off")

In [ ]:
# --- Jalankan eksperimen pada beberapa citra test ---
iou_neg = {"tanpa_neg": [], "dengan_neg": []}
contoh_neg = []

N_EKSPERIMEN = min(4, CFG["N_SAM"])

for j in range(N_EKSPERIMEN):
    x, t = ds_test[j]
    img, gm = ds_test.muat(j)
    G = torch.from_numpy(gm)
    H, W = np.array(img).shape[:2]

    emb = sam_embedding(img)   # encode sekali

    # Titik positif: 1 titik per orang (skenario ambigu)
    pts1 = [titik_dalam(mk) for mk in gm]

    # Baseline: hanya titik positif
    m_tanpa_neg, _ = sam_segmentasi(img, points=pts1, multimask=True, emb=emb)

    # Cari titik negatif otomatis dari area latar
    pts_neg = temukan_titik_negatif(gm, np.array(img).shape, n=3)

    # Dengan titik negatif
    if pts_neg:
        m_dengan_neg, _ = sam_segmentasi_neg(img, pts1, pts_neg, emb=emb, multimask=True)
    else:
        m_dengan_neg = m_tanpa_neg  # fallback jika tidak ada latar jelas

    # Hitung IoU diagonal (prediksi ke-i vs GT ke-i)
    iou_tanpa = torch.diag(iou_mask(m_tanpa_neg, G)).tolist()
    iou_dgn   = torch.diag(iou_mask(m_dengan_neg, G)).tolist()
    iou_neg["tanpa_neg"] += iou_tanpa
    iou_neg["dengan_neg"] += iou_dgn

    contoh_neg.append((img, gm, pts1, pts_neg, m_tanpa_neg, m_dengan_neg, iou_tanpa, iou_dgn))

    print(f"Citra {j+1}: IoU tanpa neg = {np.mean(iou_tanpa):.3f} "
          f"| dengan neg = {np.mean(iou_dgn):.3f} "
          f"| Δ = {np.mean(iou_dgn) - np.mean(iou_tanpa):+.3f} "
          f"| titik neg = {pts_neg}")

print(f"\nRata-rata semua citra:")
print(f"  Tanpa negatif  : {np.mean(iou_neg['tanpa_neg']):.4f}")
print(f"  Dengan negatif : {np.mean(iou_neg['dengan_neg']):.4f}")
print(f"  Delta          : {np.mean(iou_neg['dengan_neg']) - np.mean(iou_neg['tanpa_neg']):+.4f}")

In [ ]:
# --- Visualisasi per citra: tanpa neg vs dengan neg ---
fig, axes = plt.subplots(N_EKSPERIMEN, 3, figsize=(15, 5 * N_EKSPERIMEN))
if N_EKSPERIMEN == 1:
    axes = [axes]

for row, (img, gm, pts1, pts_neg, m_tanpa, m_dgn, iou_t, iou_d) in enumerate(contoh_neg):
    # Kolom 0: Ground Truth
    overlay(img, gm, judul="Ground Truth", ax=axes[row][0])

    # Kolom 1: 1 titik tanpa negatif
    overlay_dengan_prompt(
        img, m_tanpa.numpy(), pts_pos=pts1,
        judul=f"1 titik (+)\nIoU = {np.mean(iou_t):.3f}",
        ax=axes[row][1]
    )

    # Kolom 2: 1 titik + titik negatif
    overlay_dengan_prompt(
        img, m_dgn.numpy(), pts_pos=pts1, pts_neg=pts_neg,
        judul=f"1 titik (+) + {len(pts_neg)} titik (−)\nIoU = {np.mean(iou_d):.3f}",
        ax=axes[row][2]
    )

    # Anotasi: tanda ✓ atau ✗ per perubahan IoU
    for k, (a, b) in enumerate(zip(iou_t, iou_d)):
        delta = b - a
        sym = "↑" if delta > 0.01 else ("↓" if delta < -0.01 else "→")
        axes[row][2].text(5, 20 + k * 20, f"Orang {k+1}: {sym}{delta:+.2f}",
                          color="white", fontsize=8,
                          bbox=dict(facecolor="black", alpha=0.5, pad=2))

plt.suptitle("Tucil D5 — Pengaruh Titik Negatif terhadap Mask SAM\n"
             "★ = titik positif (kuning) | ✕ = titik negatif (merah)",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/js06_tucil_d5_titik_negatif.png", dpi=110, bbox_inches="tight")
plt.show()

In [ ]:
# --- Bar chart IoU: perbandingan per orang ---
fig, ax = plt.subplots(figsize=(10, 4))
n = len(iou_neg["tanpa_neg"])
x_pos = np.arange(n)
w = 0.35

ax.bar(x_pos - w/2, iou_neg["tanpa_neg"], w, label="Tanpa negatif", color="C0", alpha=0.8)
ax.bar(x_pos + w/2, iou_neg["dengan_neg"], w, label="Dengan negatif", color="C2", alpha=0.8)
ax.axhline(np.mean(iou_neg["tanpa_neg"]),  color="C0", linestyle="--", lw=1.5,
           label=f"Rata-rata tanpa neg ({np.mean(iou_neg['tanpa_neg']):.3f})")
ax.axhline(np.mean(iou_neg["dengan_neg"]), color="C2", linestyle="--", lw=1.5,
           label=f"Rata-rata dengan neg ({np.mean(iou_neg['dengan_neg']):.3f})")

ax.set_xlabel("Indeks Orang (semua citra digabung)")
ax.set_ylabel("IoU Mask")
ax.set_title("IoU per Instance: Tanpa vs Dengan Titik Negatif")
ax.legend(fontsize=8)
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/js06_tucil_d5_iou_bar.png", dpi=110, bbox_inches="tight")
plt.show()

**Bagaimana masker berubah setelah titik negatif?**

Titik negatif memberi tahu SAM “area ini bukan bagian objek”. Efeknya:
- Mask yang sebelumnya meluber ke trotoar/latar terpotong di batas tubuh
- Objek yang ambigu (misal: kaki menyatu dengan bayangan) jadi lebih ketat mengikuti kontur tubuh
- IoU cenderung naik pada kasus di mana latar belakang ikut tersegmentasi

**Mengapa 1 titik ambigu?**

SAM dilatih untuk menjawab pertanyaan “segmentasikan objek di titik ini” — tapi satu titik di torso bisa berarti tiga hal berbeda dengan IoU yang sama-sama masuk akal:
Titik di dada seseorang:
- Interpretasi A: seluruh tubuh (orang)    → mask besar
- Interpretasi B: baju/torso saja          → mask sedang  
- Interpretasi C: rok/celana + baju        → mask parsial

Karena itu SAM dengan multimask_output=True menghasilkan 3 kandidat mask dan memilih skor IoU tertinggi — tapi tetap bisa salah karena tidak ada konteks “mana batas objek yang dimaksud”.

**Bagaimana 3 titik / kotak mengurangi ambiguitas?**
| Metode / Cara | Penjelasan & Mekanisme |
| :--- | :--- |
| **3 titik** | Titik di kepala + tengah + kaki — SAM dipaksa mencakup rentang vertikal penuh tubuh, eliminasi interpretasi parsial |
| **Kotak GT** | *Bounding box* mendefinisikan batas keras objek secara eksplisit — mask hampir pasti tidak meluber keluar kotak |
| **Titik negatif** | Secara aktif menolak area latar — model tidak bisa “ragu” antara orang vs trotoar |

## Praktikum D6: Mini-Proyek Penghapus Latar Otomatis (Deteksi lalu SAM)
Aplikasi nyata: toko daring atau klinik hewan ingin memotong foto hewan dari latarnya secara otomatis.
Alur: (1) Mask R-CNN COCO mendeteksi kucing/anjing, (2) kotak terbaik dijadikan prompt SAM, (3) hasilnya disimpan sebagai PNG transparan.
Kualitas diukur dengan IoU terhadap trimap Oxford-IIIT Pet (CC BY-SA 4.0), piksel batas diabaikan.

In [ ]:
# === D6.1 Ambil beberapa foto Oxford-IIIT Pet (streaming dari server resmi, berhenti setelah N foto) ===
import tarfile
def ambil_foto_pet(root="data/pet_subset", n=8):
    os.makedirs(f"{root}/images", exist_ok=True); os.makedirs(f"{root}/trimaps", exist_ok=True)
    ann = f"{root}/annotations.tar.gz"
    if not os.path.exists(ann):
        urllib.request.urlretrieve("https://thor.robots.ox.ac.uk/pets/annotations.tar.gz", ann)
    test_ids, trimaps = set(), {}
    with tarfile.open(ann, "r:gz") as t:
        for m in t:
            if m.name.endswith("annotations/test.txt"):
                test_ids = {l.split()[0] for l in t.extractfile(m).read().decode().splitlines() if l.strip()}
            elif "/trimaps/" in m.name and m.name.endswith(".png") and "._" not in m.name:
                trimaps[os.path.basename(m.name)[:-4]] = t.extractfile(m).read()
    ada = sorted(f[:-4] for f in os.listdir(f"{root}/images") if f[:-4] in test_ids)
    if len(ada) >= n: return ada[:n]
    with urllib.request.urlopen("https://thor.robots.ox.ac.uk/pets/images.tar.gz") as r, tarfile.open(fileobj=r, mode="r|gz") as t:
        for m in t:
            iid = os.path.basename(m.name)[:-4]
            if m.name.endswith(".jpg") and iid in test_ids and iid in trimaps and iid not in ada:
                open(f"{root}/images/{iid}.jpg", "wb").write(t.extractfile(m).read())
                open(f"{root}/trimaps/{iid}.png", "wb").write(trimaps[iid]); ada.append(iid)
                if len(ada) >= n: break
    return sorted(ada)[:n]

PET = "data/pet_subset"
ID_PET = ambil_foto_pet(PET, CFG["N_PROYEK"])
print("Foto hewan:", len(ID_PET), ID_PET[:4])

In [ ]:
# === D6.2 Pipeline deteksi -> SAM -> potongan PNG transparan, dengan evaluasi IoU ===
LABEL_HEWAN = {KATEGORI.index("cat"), KATEGORI.index("dog")}
os.makedirs("hasil_potongan", exist_ok=True)

@torch.no_grad()
def potong_latar(path):
    img = Image.open(path).convert("RGB")
    out = mrcnn_coco([TF.to_tensor(img).to(DEVICE)])[0]
    pilih = [i for i, l in enumerate(out["labels"].tolist()) if l in LABEL_HEWAN]
    if not pilih:
        return img, None, None, None
    i = pilih[0]                                           # deteksi hewan dengan skor tertinggi
    m_rcnn = (out["masks"][i, 0] > 0.5).cpu()
    m_sam, _ = sam_segmentasi(img, boxes=[out["boxes"][i].tolist()])
    return img, m_rcnn, m_sam[0], float(out["scores"][i])

def iou_trimap(m, tri):
    valid = tri != 3; g = tri == 1; m = m.numpy()
    return ((m & g) & valid).sum() / max(((m | g) & valid).sum(), 1)

t0 = time.time(); rekap = []; tampil = []
for iid in ID_PET:
    img, m_rcnn, m_sam, skor = potong_latar(f"{PET}/images/{iid}.jpg")
    tri = np.array(Image.open(f"{PET}/trimaps/{iid}.png"))
    if m_sam is None:
        rekap.append((iid, None, 0.0, 0.0)); continue
    rgba = np.dstack([np.array(img), (m_sam.numpy() * 255).astype(np.uint8)])
    Image.fromarray(rgba).save(f"hasil_potongan/{iid}.png")
    rekap.append((iid, skor, iou_trimap(m_rcnn, tri), iou_trimap(m_sam, tri)))
    tampil.append((img, m_rcnn, m_sam, rgba, rekap[-1]))
WAKTU["Mini-proyek"] = time.time() - t0
print(f"{'foto':28s} {'skor det':>8s} {'IoU MaskRCNN':>12s} {'IoU SAM':>8s}")
for r in rekap:
    print(f"{r[0]:28s} {r[1] if r[1] is None else round(r[1], 3)!s:>8s} {r[2]:12.3f} {r[3]:8.3f}")
iou_r = np.mean([r[2] for r in rekap]); iou_s = np.mean([r[3] for r in rekap])
print("Rata-rata IoU  Mask R-CNN: %.4f | Mask R-CNN + SAM: %.4f | waktu %.1f s" % (iou_r, iou_s, WAKTU["Mini-proyek"]))
HASIL_PROYEK = {"IoU_MaskRCNN": float(iou_r), "IoU_MaskRCNN_SAM": float(iou_s), "n_foto": len(rekap)}

In [ ]:
# === D6.3 Visualisasi hasil mini-proyek ===
n = min(3, len(tampil))
fig, ax = plt.subplots(n, 4, figsize=(16, 4 * n), squeeze=False)
for r in range(n):
    img, m_rcnn, m_sam, rgba, rk = tampil[r]
    ax[r, 0].imshow(img); ax[r, 0].set_title(rk[0], fontsize=9)
    overlay(img, [m_rcnn.numpy()], judul=f"masker Mask R-CNN (IoU {rk[2]:.2f})", ax=ax[r, 1])
    overlay(img, [m_sam.numpy()], judul=f"masker SAM (IoU {rk[3]:.2f})", ax=ax[r, 2])
    yy, xx = np.indices(rgba.shape[:2]); kotak = np.where((((yy // 16) + (xx // 16)) % 2 == 0)[..., None], 200, 255)
    a = rgba[..., 3:] / 255.0
    ax[r, 3].imshow((a * rgba[..., :3] + (1 - a) * kotak).astype(np.uint8)); ax[r, 3].set_title("potongan PNG transparan (latar kotak-kotak)", fontsize=9)
    for c in (0, 3): ax[r, c].axis("off")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/js06_d6_miniproyek.png", dpi=100); plt.show()

In [ ]:
# === Rekap hasil dan waktu ===
with open(f"{FIG_DIR}/js06_hasil.json", "w") as f:
    json.dump({"perangkat": DEVICE, "mode": "CEPAT" if CEPAT else "PENUH", "cfg": CFG,
               "instans": {k: ringkas(v) for k, v in HASIL.items()},
               "sam_iou": HASIL_SAM, "miniproyek": HASIL_PROYEK, "waktu_detik": WAKTU}, f, indent=2)
for k, v in WAKTU.items():
    print(f"{k:35s} {v:7.1f} detik")

### **Tugas mini-proyek:** Ganti foto dengan foto hewan peliharaan Anda sendiri (unggah ke Colab), jalankan `potong_latar()`,
dan laporkan minimal 3 hasil potongan beserta analisis kegagalan (misalnya hewan tidak terdeteksi, dua hewan dalam satu foto).

In [ ]:
# === TUCIL D6: Uji potong_latar() pada foto hewan sendiri ===

try:
    from google.colab import files
    print("Unggah foto hewan Anda (jpg/png, boleh lebih dari 1):")
    uploaded = files.upload()   # muncul dialog pilih file
    FOTO_PATHS = [k for k in uploaded.keys()]
except ImportError:
    # Jika sudah upload manual atau jalankan lokal
    FOTO_PATHS = sorted(glob.glob("foto_hewan/*"))   # ubah path sesuai lokasi

print(f"Foto ditemukan: {len(FOTO_PATHS)}")
for p in FOTO_PATHS:
    print(f"  {p}")

In [ ]:
import glob, textwrap

os.makedirs("hasil_pribadi", exist_ok=True)

# --- Jalankan pipeline + catat detail kegagalan ---
def analisis_kegagalan(img, m_rcnn, m_sam, skor_det, tri=None):
    """
    Klasifikasikan jenis kegagalan berdasarkan kondisi output pipeline.
    Return: list string deskripsi masalah yang ditemukan.
    """
    masalah = []

    # 1. Tidak terdeteksi sama sekali
    if m_rcnn is None or m_sam is None:
        masalah.append("GAGAL: Tidak ada hewan terdeteksi (kucing/anjing tidak ada di output Mask R-CNN)")
        return masalah

    # 2. Skor deteksi rendah → prediksi tidak yakin
    if skor_det < 0.6:
        masalah.append(f"PERINGATAN: Skor deteksi rendah ({skor_det:.2f}) → model kurang yakin")

    W, H = img.size
    area_img = H * W
    area_mask = m_sam.float().sum().item()
    rasio = area_mask / area_img

    # 3. Mask terlalu besar → latar ikut tersegmentasi
    if rasio > 0.6:
        masalah.append(f"MASK TERLALU BESAR: {rasio:.1%} area citra → latar belakang ikut terpotong")

    # 4. Mask terlalu kecil → hewan terpotong sebagian
    if rasio < 0.03:
        masalah.append(f"MASK TERLALU KECIL: {rasio:.1%} area citra → hewan hanya terdeteksi sebagian")

    # 5. Konsistensi Mask R-CNN vs SAM
    iou_internal = iou_mask(
        m_rcnn.unsqueeze(0).float(),
        m_sam.unsqueeze(0).float()
    )[0, 0].item()
    if iou_internal < 0.5:
        masalah.append(f"INKONSISTENSI: IoU Mask R-CNN vs SAM = {iou_internal:.2f} → dua model tidak setuju area hewan")

    # 6. Evaluasi vs trimap jika tersedia
    if tri is not None:
        iou_sam_val = iou_trimap(m_sam, tri)
        if iou_sam_val < 0.7:
            masalah.append(f"IoU vs trimap rendah ({iou_sam_val:.2f}) → batas mask tidak akurat")

    if not masalah:
        masalah.append("OK: Tidak ada masalah signifikan terdeteksi")

    return masalah


rekap_pribadi = []

for path in FOTO_PATHS:
    nama = os.path.splitext(os.path.basename(path))[0]
    print(f"\n{'='*50}")
    print(f"Memproses: {nama}")

    img, m_rcnn, m_sam, skor = potong_latar(path)
    masalah = analisis_kegagalan(img, m_rcnn, m_sam, skor if skor else 0.0)

    if m_sam is not None:
        # Simpan hasil potongan transparan
        rgba = np.dstack([
            np.array(img),
            (m_sam.numpy() * 255).astype(np.uint8)
        ])
        out_path = f"hasil_pribadi/{nama}.png"
        Image.fromarray(rgba).save(out_path)

        W, H = img.size
        rasio = m_sam.float().sum().item() / (W * H)
        iou_rcnn_sam = iou_mask(
            m_rcnn.unsqueeze(0).float(),
            m_sam.unsqueeze(0).float()
        )[0, 0].item()

        rekap_pribadi.append({
            "nama": nama, "path": path,
            "skor_det": skor, "rasio_mask": rasio,
            "iou_rcnn_sam": iou_rcnn_sam,
            "img": img, "m_rcnn": m_rcnn,
            "m_sam": m_sam, "rgba": rgba,
            "masalah": masalah
        })
    else:
        rekap_pribadi.append({
            "nama": nama, "path": path,
            "skor_det": None, "rasio_mask": 0,
            "iou_rcnn_sam": 0,
            "img": img, "m_rcnn": None,
            "m_sam": None, "rgba": None,
            "masalah": masalah
        })

    for m in masalah:
        print(f"  → {m}")

In [ ]:
# --- Visualisasi: grid per foto (asli | Mask R-CNN | SAM | transparan) ---
n = len(rekap_pribadi)
fig, axes = plt.subplots(n, 4, figsize=(16, 4.5 * n), squeeze=False)

KOLOM_JUDUL = ["Asli", "Mask R-CNN", "SAM Mask", "Potongan Transparan"]
for c, judul in enumerate(KOLOM_JUDUL):
    axes[0][c].set_title(judul, fontsize=10, fontweight="bold", pad=8)

for r, rek in enumerate(rekap_pribadi):
    img = rek["img"]

    # Kolom 0: Asli
    axes[r][0].imshow(img)
    axes[r][0].set_ylabel(rek["nama"], fontsize=8, rotation=0,
                           labelpad=80, va="center")
    axes[r][0].axis("off")

    if rek["m_rcnn"] is None:
        # Kasus gagal total: tampilkan pesan
        for c in range(1, 4):
            axes[r][c].text(0.5, 0.5, "\n".join(textwrap.wrap(rek["masalah"][0], 20)),
                            ha="center", va="center", fontsize=9,
                            color="red", transform=axes[r][c].transAxes)
            axes[r][c].axis("off")
        continue

    # Kolom 1: Mask R-CNN
    overlay(img, [rek["m_rcnn"].numpy()],
            judul=f"skor={rek['skor_det']:.2f}",
            ax=axes[r][1])

    # Kolom 2: SAM
    overlay(img, [rek["m_sam"].numpy()],
            judul=f"IoU vs MaskRCNN={rek['iou_rcnn_sam']:.2f}",
            ax=axes[r][2])

    # Kolom 3: PNG transparan dengan background kotak-kotak
    rgba = rek["rgba"]
    yy, xx = np.indices(rgba.shape[:2])
    kotak = np.where((((yy // 16) + (xx // 16)) % 2 == 0)[..., None], 200, 255)
    a = rgba[..., 3:] / 255.0
    blended = (a * rgba[..., :3] + (1 - a) * kotak).astype(np.uint8)
    axes[r][3].imshow(blended)
    axes[r][3].axis("off")

    # Anotasi masalah di bawah baris
    masalah_str = " | ".join(rek["masalah"])
    fig.text(
        0.5,
        1 - (r + 1) / n + 0.01,
        f"▶ {masalah_str}",
        ha="center", fontsize=7.5,
        color="darkred" if "GAGAL" in masalah_str else
              "darkorange" if "PERINGATAN" in masalah_str else "darkgreen",
        style="italic"
    )

plt.suptitle("Mini-Proyek D6 — Pipeline Potong Latar Foto Hewan Pribadi",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/js06_tucil_d6_pribadi.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
# --- Ringkasan tabel analisis kegagalan ---
print(f"\n{'='*65}")
print(f"{'FOTO':<25} {'SKOR':>6} {'RASIO':>7} {'IoU R↔S':>8}  ANALISIS")
print(f"{'='*65}")

for rek in rekap_pribadi:
    skor_str  = f"{rek['skor_det']:.2f}" if rek["skor_det"] else "  N/A"
    rasio_str = f"{rek['rasio_mask']:.1%}" if rek["rasio_mask"] else "   N/A"
    iou_str   = f"{rek['iou_rcnn_sam']:.2f}" if rek["iou_rcnn_sam"] else "  N/A"
    status    = rek["masalah"][0][:35]
    print(f"{rek['nama']:<25} {skor_str:>6} {rasio_str:>7} {iou_str:>8}  {status}")

print(f"{'='*65}")
n_ok    = sum(1 for r in rekap_pribadi if "OK" in r["masalah"][0])
n_gagal = sum(1 for r in rekap_pribadi if "GAGAL" in r["masalah"][0])
n_warn  = len(rekap_pribadi) - n_ok - n_gagal
print(f"\nRingkasan: {n_ok} berhasil | {n_warn} ada peringatan | {n_gagal} gagal total")
print(f"Foto berhasil dipotong disimpan di: hasil_pribadi/")